<a href="https://colab.research.google.com/github/ted-chang80/AIFFEL_quest_eng/blob/main/LLM_Application/LLM01/%EB%B2%88%EC%97%AD%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python3 -m pip install --upgrade pip
!python3 -m pip install konlpy # Python 3.x



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 28.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [konlpy]


In [2]:
# Install MeCab for KoNLPy using the official KoNLPy script
!apt-get update
!apt-get install g++ openjdk-8-jdk python3-dev python3-pip curl
!python3 -m pip install konlpy
!python3 -m pip install JPype1
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

# Note: konlpy.tag.Mecab often fails with 'NameError: name Tagger is not defined'
# due to system library path issues in Colab.
# Using 'python-mecab-ko' is a much more stable alternative.

try:
    from konlpy.tag import Mecab
    mecab = Mecab()
    print("KoNLPy Mecab initialized successfully.")
except Exception as e:
    print(f"KoNLPy Mecab failed: {e}")
    print("Attempting to use stable alternative: python-mecab-ko")
    !pip install python-mecab-ko
    from mecab import MeCab
    mecab = MeCab()
    print("Alternative MeCab (python-mecab-ko) initialized successfully.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/univers

In [3]:
# 기존 konlpy MeCab 대신 더 안정적인 python-mecab-ko 설치 및 사용
!pip install python-mecab-ko

from mecab import MeCab
mecab = MeCab()

try:
    test_tokens = mecab.morphs('새로운 라이브러리로 형태소 분석을 테스트합니다.')
    print("MeCab 로드 성공!:", test_tokens)
except Exception as e:
    print(f"오류 발생: {e}")

MeCab 로드 성공!: ['새로운', '라이브러리', '로', '형태소', '분석', '을', '테스트', '합니다', '.']


In [4]:
!pip install sentencepiece nltk

In [5]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.10.0+cu128


In [6]:
data_url = 'https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv'
!wget -O /content/ChatbotData.csv {data_url}

--2026-05-15 07:19:14--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘/content/ChatbotData.csv’

/content/ChatbotDat 100%[===================>] 868.99K  --.-KB/s    in 0.04s   

2026-05-15 07:19:14 (19.1 MB/s) - ‘/content/ChatbotData.csv’ saved [889842/889842]



In [35]:
!unzip -o /content/ko.zip -d /content/

Archive:  /content/ko.zip
  inflating: /content/ko.bin         
  inflating: /content/ko.tsv         


In [37]:
df = pd.read_csv('/content/ChatbotData.csv')
display(df.head())

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [114]:
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

In [115]:
def advanced_preprocess(sentence):
    # 1. 한글, 숫자, 영문, 주요 문장부호 제외하고 제거
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,~]", " ", sentence)

    # 2. 문장 부호 양옆에 공백 추가 (토큰화 도움)
    sentence = re.sub(r"([?.!,~])", r" \1 ", sentence)

    # 3. 반복되는 자음/모음 정규화 (예: ㅋㅋㅋㅋ -> ㅋㅋ)
    sentence = re.sub(r"([ㄱ-ㅎㅏ-ㅣ])\1+", r"\1\1", sentence)

    # 4. 다중 공백 제거
    sentence = re.sub(r"[ ]+", " ", sentence)

    return sentence.strip()

# 정제 적용
df['Q_refined'] = df['Q'].apply(advanced_preprocess)
df['A_refined'] = df['A'].apply(advanced_preprocess)

# 너무 짧은 문장 제외 (토큰 2개 미만 등)
def is_valid_length(s, min_len=2):
    return len(mecab.morphs(s)) >= min_len

mask = df.apply(lambda row: is_valid_length(row['Q_refined']) and is_valid_length(row['A_refined']), axis=1)
df_filtered = df[mask]

print(f"정제 전 데이터 수: {len(df)}")
print(f"정제 후 데이터 수: {len(df_filtered)}")
display(df_filtered[['Q_refined', 'A_refined']].head())

정제 전 데이터 수: 11750
정제 후 데이터 수: 11630


,Q_refined,A_refined
0,12시 땡 !,하루가 또 가네요 .
1,1지망 학교 떨어졌어,위로해 드립니다 .
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠 .
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠 .
4,ppl 심하네,눈살이 찌푸려지죠 .


In [116]:
df['Q'] = df['Q'].apply(preprocess_sentence)
df['A'] = df['A'].apply(preprocess_sentence)
display(df.head())

,Q,A,label,Q_refined,A_refined
0,12시 땡!,하루가 또 가네요.,0,12시 땡 !,하루가 또 가네요 .
1,1지망 학교 떨어졌어,위로해 드립니다.,0,1지망 학교 떨어졌어,위로해 드립니다 .
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0,3박4일 놀러가고 싶다,여행은 언제나 좋죠 .
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠 .
4,ppl 심하네,눈살이 찌푸려지죠.,0,ppl 심하네,눈살이 찌푸려지죠 .


In [117]:
# konlpy.tag.Mecab 대신 더 안정적인 python-mecab-ko 라이브러리 사용
from mecab import MeCab
mecab = MeCab()

MAX_LEN = 40 # Set maximum length for tokenized sentences

In [118]:
# Re-execute the definition of MAX_LEN, as it was in the cell that previously failed
MAX_LEN = 40 # Set maximum length for tokenized sentences

In [119]:
def build_corpus(src_data, tgt_data, tokenizer, max_len):
    tokenized_questions = []
    tokenized_answers = []

    for q, a in zip(src_data, tgt_data):
        processed_q = preprocess_sentence(q)
        processed_a = preprocess_sentence(a)

        tokens_q = tokenizer(processed_q)
        tokens_a = tokenizer(processed_a)

        if len(tokens_q) <= max_len and len(tokens_a) <= max_len:
            tokenized_questions.append(tokens_q)
            tokenized_answers.append(tokens_a)

    return tokenized_questions, tokenized_answers

# 중복 제거 및 코퍼스 생성 재실행
initial_rows = len(df)
df.drop_duplicates(subset=['Q', 'A'], inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate Q-A pairs.")

# Build the corpus using the successfully initialized mecab
que_corpus, ans_corpus = build_corpus(df['Q'], df['A'], mecab.morphs, MAX_LEN)

print(f"\nNumber of questions in corpus: {len(que_corpus)}")
print(f"Number of answers in corpus: {len(ans_corpus)}")

print("\nExample question-answer pairs from the corpus:")
for i in range(min(5, len(que_corpus))):
    print(f"Q: {' '.join(que_corpus[i])}")
    print(f"A: {' '.join(ans_corpus[i])}")

Removed 0 duplicate Q-A pairs.

Number of questions in corpus: 11750
Number of answers in corpus: 11750

Example question-answer pairs from the corpus:
Q: 12 시 땡 !
A: 하루 가 또 가 네요 .
Q: 1 지망 학교 떨어졌 어
A: 위로 해 드립니다 .
Q: 3 박 4 일 놀 러 가 고 싶 다
A: 여행 은 언제나 좋 죠 .
Q: 3 박 4 일 정도 놀 러 가 고 싶 다
A: 여행 은 언제나 좋 죠 .
Q: ppl 심하 네
A: 눈살 이 찌푸려 지 죠 .


In [120]:
print('Missing values before dropping:')
display(df.isnull().sum())

df.dropna(inplace=True)
print('\nMissing values after dropping:')
display(df.isnull().sum())

Missing values before dropping:


,0
Q,0
A,0
label,0
Q_refined,0
A_refined,0



Missing values after dropping:


,0
Q,0
A,0
label,0
Q_refined,0
A_refined,0


In [121]:
# Remove duplicate Q-A pairs from the DataFrame before building corpus
initial_rows = len(df)
df.drop_duplicates(subset=['Q', 'A'], inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate Q-A pairs.")

# Build the corpus
que_corpus, ans_corpus = build_corpus(df['Q'], df['A'], mecab.morphs, MAX_LEN)

print(f"\nNumber of questions in corpus: {len(que_corpus)}")
print(f"Number of answers in corpus: {len(ans_corpus)}")

print("\nExample question-answer pairs from the corpus:")
for i in range(5):
    print(f"Q: {' '.join(que_corpus[i])}")
    print(f"A: {' '.join(ans_corpus[i])}")

Removed 0 duplicate Q-A pairs.

Number of questions in corpus: 11750
Number of answers in corpus: 11750

Example question-answer pairs from the corpus:
Q: 12 시 땡 !
A: 하루 가 또 가 네요 .
Q: 1 지망 학교 떨어졌 어
A: 위로 해 드립니다 .
Q: 3 박 4 일 놀 러 가 고 싶 다
A: 여행 은 언제나 좋 죠 .
Q: 3 박 4 일 정도 놀 러 가 고 싶 다
A: 여행 은 언제나 좋 죠 .
Q: ppl 심하 네
A: 눈살 이 찌푸려 지 죠 .


In [122]:
print('--- 코퍼스 데이터 예시 (토큰화 결과) ---\n')
for i in range(5):
    print(f'Sample {i+1}')
    print(f'Question Tokenized: {que_corpus[i]}')
    print(f'Answer Tokenized:   {ans_corpus[i]}')
    print('-' * 40)

--- 코퍼스 데이터 예시 (토큰화 결과) ---

Sample 1
Question Tokenized: ['12', '시', '땡', '!']
Answer Tokenized:   ['하루', '가', '또', '가', '네요', '.']
----------------------------------------
Sample 2
Question Tokenized: ['1', '지망', '학교', '떨어졌', '어']
Answer Tokenized:   ['위로', '해', '드립니다', '.']
----------------------------------------
Sample 3
Question Tokenized: ['3', '박', '4', '일', '놀', '러', '가', '고', '싶', '다']
Answer Tokenized:   ['여행', '은', '언제나', '좋', '죠', '.']
----------------------------------------
Sample 4
Question Tokenized: ['3', '박', '4', '일', '정도', '놀', '러', '가', '고', '싶', '다']
Answer Tokenized:   ['여행', '은', '언제나', '좋', '죠', '.']
----------------------------------------
Sample 5
Question Tokenized: ['ppl', '심하', '네']
Answer Tokenized:   ['눈살', '이', '찌푸려', '지', '죠', '.']
----------------------------------------


In [123]:
def build_corpus(src_data, tgt_data, tokenizer, max_len):
    tokenized_questions = []
    tokenized_answers = []

    # Remove duplicate rows from the DataFrame first to preserve Q-A pair integrity
    # The previous cleaning steps already handled general duplicates in df, but re-applying to be safe.
    # However, the input to build_corpus here is already Series, so we'll assume df itself is cleaned.

    for q, a in zip(src_data, tgt_data):
        # Preprocess sentences
        processed_q = preprocess_sentence(q)
        processed_a = preprocess_sentence(a)

        # Tokenize sentences
        tokens_q = tokenizer(processed_q)
        tokens_a = tokenizer(processed_a)

        # Filter by maximum length
        if len(tokens_q) <= max_len and len(tokens_a) <= max_len:
            tokenized_questions.append(tokens_q)
            tokenized_answers.append(tokens_a)

    return tokenized_questions, tokenized_answers

In [124]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. 특수 토큰 정의
# <sos>: Start of Sequence, <eos>: End of Sequence
# <pad>는 보통 0번으로 지정됩니다.

def tokenize(corpus):
    # 모든 단어를 포함하되, 빈도수가 낮은 단어는 제외하고 싶다면 num_words를 조절할 수 있습니다.
    tokenizer = Tokenizer(filters='', lower=False, oov_token='<unk>')
    tokenizer.fit_on_texts(corpus)

    tensor = tokenizer.texts_to_sequences(corpus)
    tensor = pad_sequences(tensor, padding='post', maxlen=MAX_LEN)

    return tensor, tokenizer

# 2. 질문과 답변 코퍼스에 대해 각각 인코딩 진행
# 답변(Target) 데이터에는 시작과 끝을 알리는 토큰이 필요할 수 있으나,
# 여기서는 우선 전체 코퍼스에 대한 기본 토큰화를 진행합니다.

# 질문 코퍼스 인코딩
que_tensor, que_tokenizer = tokenize(que_corpus)

# 답변 코퍼스 인코딩
ans_tensor, ans_tokenizer = tokenize(ans_corpus)

print("질문 텐서 크기:", que_tensor.shape)
print("답변 텐서 크기:", ans_tensor.shape)

# 샘플 확인
print("\n인코딩된 질문 샘플 (첫 번째 문장):\n", que_tensor[0])
print("\n단어 사전 크기 (질문):", len(que_tokenizer.word_index))
print("단어 사전 크기 (답변):", len(ans_tokenizer.word_index))

질문 텐서 크기: (11750, 40)
답변 텐서 크기: (11750, 40)

인코딩된 질문 샘플 (첫 번째 문장):
 [1442  723 2180  140    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]

단어 사전 크기 (질문): 5318
단어 사전 크기 (답변): 3899


### Step 4. Augmentation (Lexical Substitution)

사전 학습된 Word2Vec 모델을 사용하여 유사어 치환 기반의 데이터 증강을 수행합니다.

In [125]:
!pip install gensim
# Note: If ko.zip is not manually uploaded, the following line will fail.
# I will proceed with a check in the next cell to prevent crashes.
!unzip -o /content/ko.zip -d /content/ || echo "ko.zip not found. Proceeding without Lexical Substitution."

Archive:  /content/ko.zip
  inflating: /content/ko.bin         
  inflating: /content/ko.tsv         


In [126]:
import gensim
import random
import warnings
import os
import pickle
from gensim.models import KeyedVectors, Word2Vec
warnings.filterwarnings('ignore')

word2vec = None
model_path = '/content/ko.bin'

try:
    # Attempt 1: Standard load
    word2vec = Word2Vec.load(model_path)
    print("Successfully loaded using Word2Vec.load!")
except Exception as e:
    try:
        # Attempt 2: Load with latin1 encoding (fix for Gensim 3.x models in Python 3)
        with open(model_path, 'rb') as f:
            word2vec = pickle.load(f, encoding='latin1')
        print("Successfully loaded using pickle with latin1 encoding!")
    except Exception as e2:
        try:
            # Attempt 3: KeyedVectors binary format
            word2vec = KeyedVectors.load_word2vec_format(model_path, binary=True, unicode_errors='replace')
            print("Successfully loaded using load_word2vec_format!")
        except Exception as e3:
            print(f"All attempts failed.\n1: {e}\n2: {e2}\n3: {e3}")

def lexical_sub(sentence, model):
    if model is None:
        return sentence

    res = []
    # Handle both Word2Vec objects and KeyedVectors objects
    wv = model.wv if hasattr(model, 'wv') else model

    for tok in sentence:
        try:
            similar_words = wv.most_similar(tok, topn=5)
            res.append(random.choice(similar_words)[0])
        except:
            res.append(tok)
    return res

ERROR:gensim.models.word2vec:Model load error. Was model saved using code from an older Gensim Version? Try loading older model using gensim-3.8.3, then re-saving, to restore compatibility with current code.


Successfully loaded using pickle with latin1 encoding!


In [127]:
from tqdm.notebook import tqdm

# Re-run augmentation with the loaded model
aug_que_corpus = []
aug_ans_corpus = []

if word2vec is not None:
    print('Augmenting questions using Word2Vec...')
    for q, a in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus)):
        aug_que_corpus.append(lexical_sub(q, word2vec))
        aug_ans_corpus.append(a)

    print('Augmenting answers using Word2Vec...')
    for q, a in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus)):
        aug_que_corpus.append(q)
        aug_ans_corpus.append(lexical_sub(a, word2vec))

    full_que_corpus = que_corpus + aug_que_corpus
    full_ans_corpus = ans_corpus + aug_ans_corpus

    print(f'\nAugmentation Process Finished!')
    print(f'Total Data Size: {len(full_que_corpus)}')
    print(f"Original sample: {' '.join(que_corpus[10])}")
    print(f"Augmented sample: {' '.join(full_que_corpus[len(que_corpus) + 10])}")
else:
    print("Word2Vec model is not loaded. Skipping augmentation.")

Augmenting questions using Word2Vec...


  0%|          | 0/11750 [00:00<?, ?it/s]

Augmenting answers using Word2Vec...


  0%|          | 0/11750 [00:00<?, ?it/s]


Augmentation Process Finished!
Total Data Size: 35250
Original sample: sns 보 면 나 만 빼 고 다 행복 해 보여
Augmented sample: sns 보 면 나 만 빼 고 다 행복 해 보여


In [128]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def finalize_corpus(que_corpus, ans_corpus):
    # Add <sos> and <eos> to answers
    final_que = que_corpus
    final_ans = []

    for ans in ans_corpus:
        final_ans.append(['<sos>'] + ans + ['<eos>'])

    return final_que, final_ans

# Prepare corpus with special tokens
que_corpus_final, ans_corpus_final = finalize_corpus(full_que_corpus, full_ans_corpus)

# Tokenization and Padding
def tokenize_and_pad(corpus, max_len):
    tokenizer = Tokenizer(filters='', lower=False, oov_token='<unk>')
    tokenizer.fit_on_texts(corpus)
    tensor = tokenizer.texts_to_sequences(corpus)
    tensor = pad_sequences(tensor, padding='post', maxlen=max_len)
    return tensor, tokenizer

# Generate final tensors
# We allow max_len + 2 for answers to accommodate <sos> and <eos>
que_train, que_tokenizer = tokenize_and_pad(que_corpus_final, MAX_LEN)
ans_train, ans_tokenizer = tokenize_and_pad(ans_corpus_final, MAX_LEN + 2)

print(f"Final Question Tensor Shape: {que_train.shape}")
print(f"Final Answer Tensor Shape: {ans_train.shape}")
print(f"Question Vocab Size: {len(que_tokenizer.word_index)}")
print(f"Answer Vocab Size: {len(ans_tokenizer.word_index)}")

# Sample check
print('\nSample Answer with tokens:', ans_corpus_final[0])
print('Vectorized Answer:', ans_train[0])

Final Question Tensor Shape: (35250, 40)
Final Answer Tensor Shape: (35250, 42)
Question Vocab Size: 5318
Answer Vocab Size: 3901

Sample Answer with tokens: ['<sos>', '하루', '가', '또', '가', '네요', '.', '<eos>']
Vectorized Answer: [  2 284  19 295  19  23   4   3   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0]


In [129]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
print("슝=3")

슝=3


In [130]:
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    enc_mask = generate_padding_mask(src)
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    dec_enc_mask = generate_padding_mask(src)

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

슝=3


In [131]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

print("슝=3")

슝=3


In [132]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        out = self.fc2(out)          # 두 번째 Dense
        return out

print("슝=3")

슝=3


In [133]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # nn.LayerNorm은 마지막 차원(d_model)을 기준으로 정규화
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention 단계
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual  # residual connection

        # Position-Wise Feed Forward 단계
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual  # residual connection

        return out, enc_attn

print("슝=3")

슝=3


In [134]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn

print("슝=3")

슝=3


In [135]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        out = self.final_norm(out)
        return out, enc_attns

print("슝=3")

슝=3


In [136]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        out = self.final_norm(out)
        return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [137]:
import math

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))

        self.do = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [138]:
# 질문과 답변의 단어 사전 크기 정의 (이전 단계의 변수 활용)
SRC_VOCAB_SIZE = len(que_tokenizer.word_index) + 1
TGT_VOCAB_SIZE = len(ans_tokenizer.word_index) + 1

# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True if SRC_VOCAB_SIZE == TGT_VOCAB_SIZE else False)

transformer = transformer.to(device)

d_model = 512

print("모델 생성 성공! 슝=3")

모델 생성 성공! 슝=3


In [139]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60): # 4000
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

print("슝=3")

슝=3


In [140]:
# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)

print("슝=3")

슝=3


In [141]:
def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

print("슝=3")

슝=3


In [142]:
def train_step(src, tgt, model, optimizer):
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [143]:
from torch.utils.data import DataLoader, TensorDataset

# 하이퍼파라미터
D_MODEL = 256
NUM_LAYERS = 2
NUM_HEADS = 8
D_FF = 512
DROPOUT = 0.1
BATCH_SIZE = 64

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 데이터셋 구성
# que_train과 ans_train은 numpy 배열 상태이므로 torch tensor로 변환
que_dataset = torch.LongTensor(que_train)
ans_dataset = torch.LongTensor(ans_train)

# ans_train에서 입력(sos 포함, eos 제외)과 타겟(sos 제외, eos 포함) 분리
ans_input = ans_dataset[:, :-1]
ans_target = ans_dataset[:, 1:]

train_dataset = TensorDataset(que_dataset, ans_input, ans_target)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 모델 인스턴스 생성
src_vocab_size = len(que_tokenizer.word_index) + 1
tgt_vocab_size = len(ans_tokenizer.word_index) + 1

transformer = Transformer(
    n_layers=NUM_LAYERS,
    d_model=D_MODEL,
    n_heads=NUM_HEADS,
    d_ff=D_FF,
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    pos_len=MAX_LEN + 2, # 패딩을 고려한 넉넉한 길이
    dropout=DROPOUT
).to(device)

print(f"모델 생성 완료! (Device: {device})")

모델 생성 완료! (Device: cuda)


In [144]:
import time

# Loss Function & Optimizer
# Ignore index 0 (padding token)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

def train_step(src, tgt_input, tgt_output):
    transformer.train()

    # Generate masks
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_input)

    # Forward pass
    optimizer.zero_grad()
    logits, _, _, _ = transformer(src, tgt_input, enc_mask, dec_enc_mask, dec_mask)

    # Compute loss
    # logits: [batch, seq_len, vocab], tgt_output: [batch, seq_len]
    loss = criterion(logits.view(-1, tgt_vocab_size), tgt_output.view(-1))

    # Backward pass
    loss.backward()
    optimizer.step()

    return loss.item()

# Training Loop
EPOCHS = 20

print("학습 시작...")
for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0

    for batch, (src, tgt_in, tgt_out) in enumerate(train_loader):
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

        loss = train_step(src, tgt_in, tgt_out)
        total_loss += loss

    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f} | Time: {time.time() - start:.2f}s")

print("학습 완료!")

학습 시작...
Epoch 1 Loss: 67.1651 | Time: 23.88s
Epoch 2 Loss: 10.7558 | Time: 16.63s
Epoch 3 Loss: 7.5750 | Time: 16.27s
Epoch 4 Loss: 6.5641 | Time: 16.39s
Epoch 5 Loss: 6.0482 | Time: 18.30s
Epoch 6 Loss: 5.6982 | Time: 17.80s
Epoch 7 Loss: 5.4564 | Time: 16.62s
Epoch 8 Loss: 5.2577 | Time: 16.48s
Epoch 9 Loss: 5.0966 | Time: 17.00s
Epoch 10 Loss: 4.9437 | Time: 16.53s
Epoch 11 Loss: 4.8131 | Time: 16.84s
Epoch 12 Loss: 4.6878 | Time: 16.63s
Epoch 13 Loss: 4.5655 | Time: 20.39s
Epoch 14 Loss: 4.4479 | Time: 16.56s
Epoch 15 Loss: 4.3411 | Time: 16.62s
Epoch 16 Loss: 4.2346 | Time: 16.56s
Epoch 17 Loss: 4.1279 | Time: 16.56s
Epoch 18 Loss: 4.0194 | Time: 16.59s
Epoch 19 Loss: 3.9164 | Time: 17.02s
Epoch 20 Loss: 3.8131 | Time: 16.50s
학습 완료!


### Step 5. 모델 평가 (Inference)

학습된 모델을 사용하여 새로운 질문에 대한 답변을 생성하는 함수를 구현합니다.

In [145]:
def evaluate(sentence, model, src_tokenizer, tgt_tokenizer):
    model.eval()

    # 1. 입력 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)

    # 2. 인덱스 변환 및 패딩
    inputs = src_tokenizer.texts_to_sequences([tokens])
    inputs = pad_sequences(inputs, maxlen=MAX_LEN, padding='post')
    inputs = torch.LongTensor(inputs).to(device)

    # 3. 디코더 입력 초기화 (<sos> 토큰)
    sos_token = tgt_tokenizer.word_index['<sos>']
    eos_token = tgt_tokenizer.word_index['<eos>']
    output = torch.tensor([[sos_token]], dtype=torch.long).to(device)

    # 4. 루프를 돌며 다음 토큰 예측
    for i in range(MAX_LEN + 2):
        enc_mask, dec_enc_mask, dec_mask = generate_masks(inputs, output)

        with torch.no_grad():
            predictions, _, _, _ = model(inputs, output, enc_mask, dec_enc_mask, dec_mask)

        # 마지막 시점의 예측값 추출
        predictions = predictions[:, -1, :]
        predicted_id = torch.argmax(predictions, dim=-1).item()

        # <eos> 토큰이 나오면 종료
        if predicted_id == eos_token:
            break

        # 예측된 토큰을 디코더 입력에 추가
        output = torch.cat([output, torch.tensor([[predicted_id]], device=device)], dim=-1)

    # 5. 인덱스를 단어로 변환
    result_indices = output.squeeze().tolist()[1:] # <sos> 제외
    result_words = [tgt_tokenizer.index_word[idx] for idx in result_indices]

    return ' '.join(result_words)

In [146]:
test_sentences = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

print("--- Chatbot Inference Results ---\n")
for sentence in test_sentences:
    response = evaluate(sentence, transformer, que_tokenizer, ans_tokenizer)
    print(f"Q: {sentence}")
    print(f"A: {response}\n")
    print(f"[Model Hyperparameters]")
print(f"- n_layers: {NUM_LAYERS}")
print(f"- d_model: {D_MODEL}")
print(f"- n_heads: {NUM_HEADS}")
print(f"- d_ff: {D_FF}")
print(f"- dropout: {DROPOUT}")
print(f"- src_vocab_size: {SRC_VOCAB_SIZE}")
print(f"- tgt_vocab_size: {TGT_VOCAB_SIZE}")

print(f"\n[Training Parameters]")
print(f"- BATCH_SIZE: {BATCH_SIZE}")
print(f"- TOTAL_EPOCHS: {EPOCHS}")
print(f"- MAX_LEN: {MAX_LEN}")
print(f"- DEVICE: {device}")

# 스케줄러 설정 정보
print(f"- Warmup Steps: {learning_rate.warmup_steps}")

--- Chatbot Inference Results ---

Q: 지루하다, 놀러가고 싶어.
A: 마음 이 네요 .

Q: 오늘 일찍 일어났더니 피곤하다.
A: 사랑 은 사랑 은 사랑 이 네요 .

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 마음 이 네요 .

Q: 집에 있는다는 소리야.
A: 마음 이 네요 .



In [149]:
def beam_search_decoder(sentence, model, src_tokenizer, tgt_tokenizer, beam_size=3):
    model.eval()
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    inputs = src_tokenizer.texts_to_sequences([tokens])
    inputs = pad_sequences(inputs, maxlen=MAX_LEN, padding='post')
    inputs = torch.LongTensor(inputs).to(device)

    sos_token = tgt_tokenizer.word_index['<sos>']
    eos_token = tgt_tokenizer.word_index['<eos>']

    # (score, sequence)
    beams = [(0, [sos_token])]

    for _ in range(MAX_LEN):
        new_beams = []
        for score, seq in beams:
            if seq[-1] == eos_token:
                new_beams.append((score, seq))
                continue

            out_tensor = torch.LongTensor([seq]).to(device)
            enc_mask, dec_enc_mask, dec_mask = generate_masks(inputs, out_tensor)

            with torch.no_grad():
                logits, _, _, _ = model(inputs, out_tensor, enc_mask, dec_enc_mask, dec_mask)

            probs = F.log_softmax(logits[:, -1, :], dim=-1)
            top_probs, top_ids = probs.topk(beam_size)

            for i in range(beam_size):
                new_beams.append((score + top_probs[0][i].item(), seq + [top_ids[0][i].item()]))

        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
        if all(seq[-1] == eos_token for _, seq in beams): break

    best_seq = beams[0][1]
    result_indices = [idx for idx in best_seq if idx not in [sos_token, eos_token]]
    return ' '.join([tgt_tokenizer.index_word[idx] for idx in result_indices])

print("--- Beam Search Inference Results ---\n")
for sentence in test_sentences:
    response = beam_search_decoder(sentence, transformer_refined, que_tokenizer_refined, ans_tokenizer_refined)
    print(f"Q: {sentence}")
    print(f"A: {response}\n")
    print(f"[Model Hyperparameters]")
print(f"- n_layers: {NUM_LAYERS}")
print(f"- d_model: {D_MODEL}")
print(f"- n_heads: {NUM_HEADS}")
print(f"- d_ff: {D_FF}")
print(f"- dropout: {DROPOUT}")
print(f"- src_vocab_size: {SRC_VOCAB_SIZE}")
print(f"- tgt_vocab_size: {TGT_VOCAB_SIZE}")

print(f"\n[Training Parameters]")
print(f"- BATCH_SIZE: {BATCH_SIZE}")
print(f"- TOTAL_EPOCHS: {EPOCHS}")
print(f"- MAX_LEN: {MAX_LEN}")
print(f"- DEVICE: {device}")

# 스케줄러 설정 정보
print(f"- Warmup Steps: {learning_rate.warmup_steps}")

--- Beam Search Inference Results ---

Q: 지루하다, 놀러가고 싶어.
A: . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

[Model Hyperparameters]
Q: 오늘 일찍 일어났더니 피곤하다.
A: . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

[Model Hyperparameters]
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

[Model Hyperparameters]
Q: 집에 있는다는 소리야.
A: . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

[Model Hyperparameters]
- n_layers: 4
- d_model: 512
- n_heads: 8
- d_ff: 2048
- dropout: 0.1
- src_vocab_size: 5319
- tgt_vocab_size: 3902

[Training Parameters]
- BATCH_SIZE: 64
- TOTAL_EPOCHS: 20
- MAX_LEN: 40
- DEVICE: cuda
- Warmup Steps: 4000


In [151]:
# 1. 토크나이저 인덱스 매핑 재확인
print("--- Tokenizer Index Verification ---")
def print_special_tokens(tokenizer, name):
    print(f"[{name}]")
    for token in ['<sos>', '<eos>', '<unk>', '.', '!']:
        idx = tokenizer.word_index.get(token, 'Not Found')
        print(f"  {token}: {idx}")

print_special_tokens(que_tokenizer, 'Question Tokenizer')
print_special_tokens(ans_tokenizer, 'Answer Tokenizer')

# 2. 보정된 추론 로직 (에러 수정 버전)
def evaluate_v3(sentence, model, src_tokenizer, tgt_tokenizer):
    model.eval()
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    inputs = src_tokenizer.texts_to_sequences([tokens])
    inputs = pad_sequences(inputs, maxlen=MAX_LEN, padding='post')
    inputs = torch.LongTensor(inputs).to(device)

    sos_token = tgt_tokenizer.word_index['<sos>']
    eos_token = tgt_tokenizer.word_index['<eos>']
    dot_token = tgt_tokenizer.word_index.get('.', -1)

    output = torch.tensor([[sos_token]], dtype=torch.long).to(device)

    for i in range(MAX_LEN):
        enc_mask, dec_enc_mask, dec_mask = generate_masks(inputs, output)
        with torch.no_grad():
            predictions, _, _, _ = model(inputs, output, enc_mask, dec_enc_mask, dec_mask)

        logits = predictions[0, -1, :]

        if i == 0:
            top_values, top_indices = torch.topk(logits, 5)
            predicted_id = top_indices[0].item()
            # 마침표나 종료 토큰이 첫 번째로 나올 경우 차순위 선택
            if predicted_id == dot_token or predicted_id == eos_token:
                predicted_id = top_indices[1].item()
        else:
            predicted_id = torch.argmax(logits, dim=-1).item()

        if predicted_id == eos_token: break
        output = torch.cat([output, torch.tensor([[predicted_id]], device=device)], dim=-1)

    # result_indices 처리 시 리스트 차원 유지
    indices = output.squeeze().detach().cpu().numpy()
    if indices.ndim == 0: # 단일 값일 경우 처리
        result_indices = []
    else:
        result_indices = indices.tolist()[1:]

    return ' '.join([tgt_tokenizer.index_word.get(idx, '') for idx in result_indices])

print("\n--- Improved Inference Test (v3) ---")
for sentence in test_sentences:
    res = evaluate_v3(sentence, transformer, que_tokenizer, ans_tokenizer)
    print(f"Q: {sentence}\nA: {res}\n")

--- Tokenizer Index Verification ---
[Question Tokenizer]
  <sos>: Not Found
  <eos>: Not Found
  <unk>: 1
  .: 3
  !: 128
[Answer Tokenizer]
  <sos>: 2
  <eos>: 3
  <unk>: 1
  .: 4
  !: 69

--- Improved Inference Test ---


TypeError: 'int' object is not subscriptable